# Two-Stock Trinomial Model

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime

def get_adjusted_prices(tickers, start="2020-01-01", end=None):
    """
    Download adjusted closing prices for exactly two tickers.

    Parameters
    ----------
    tickers : list[str]
        List of two ticker symbols.
    start : str, optional
        Start date for historical data (YYYY-MM-DD).
    end : str, optional
        End date for historical data.

    Returns
    -------
    pd.DataFrame
        DataFrame with two columns (one per ticker) of adjusted prices.

    Raises
    ------
    ValueError
        If adjusted or closing prices cannot be found or if the number of tickers is not two.
    """
    data = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
        group_by="column"
    )

    if isinstance(data.columns, pd.MultiIndex):
        # Multi-ticker download
        if "Adj Close" in data.columns.levels[0]:
            prices = data["Adj Close"].copy()
        elif "Close" in data.columns.levels[0]:
            prices = data["Close"].copy()
        else:
            raise ValueError("Could not find Close/Adj Close in downloaded data.")
    else:
        # Single ticker fallback
        if "Adj Close" in data.columns:
            prices = data[["Adj Close"]].copy()
        elif "Close" in data.columns:
            prices = data[["Close"]].copy()
        else:
            raise ValueError("Could not find Close/Adj Close in downloaded data.")
        prices.columns = tickers

    prices = prices.dropna()
    if prices.shape[1] != 2:
        raise ValueError("Expected exactly two tickers.")
    return prices

def get_closest_expiration(expirations, target_days=365):
    """
    Select the option expiration closest to a target number of days from today.

    Parameters
    ----------
    expirations : list[str]
        List of expiration dates as strings.
    target_days : int
        Target time to maturity in days.

    Returns
    -------
    str
        Expiration date closest to the target horizon.
    """
    today = datetime.today()

    exp_dates = pd.to_datetime(expirations)
    days = [(exp - today).days for exp in exp_dates]

    # find closest to target_days
    idx = np.argmin([abs(d - target_days) for d in days])

    return expirations[idx]

def get_strike_price(ticker, option_type="put", target="ATM"):
    """
    Retrieve a representative strike and price for a given ticker's option.

    Parameters
    ----------
    ticker : str
        Stock ticker symbol.
    option_type : str
        'put' or 'call'.
    target : str
        'ATM' for at-the-money or 'OTM' for slightly out-of-the-money.

    Returns
    -------
    dict
        Dictionary containing strike, option price, expiration, and current stock price.
    """
    tk = yf.Ticker(ticker)

    # Use closest expiration
    exp = get_closest_expiration(tk.options, target_days=365)

    chain = tk.option_chain(exp)
    options = chain.puts if option_type == "put" else chain.calls

    # Current price
    S0 = tk.history(period="1d")["Close"].iloc[-1]

    # Find closest strike
    options["distance"] = np.abs(options["strike"] - S0)

    if target == "ATM":
        row = options.sort_values("distance").iloc[0]

    elif target == "OTM":
        # slightly out-of-the-money put (strike < price)
        otm = options[options["strike"] < S0]
        row = otm.sort_values("distance").iloc[0]

    else:
        raise ValueError("target must be 'ATM' or 'OTM'")

    return {
        "strike": float(row["strike"]),
        "price": float(row["lastPrice"]),
        "expiration": exp,
        "S0": float(S0)
    }

In [ ]:
def estimate_moments(prices, periods_per_year=252):
    """
    Estimate annualized mean returns, volatility, and correlation from price data.

    Parameters
    ----------
    prices : pd.DataFrame
        DataFrame of price series for two assets.
    periods_per_year : int
        Number of periods per year (e.g., 252 for daily data).

    Returns
    -------
    dict
        Contains annualized means (mu1, mu2), volatilities (sigma1, sigma2),
        correlation (rho), and log return series.
    """
    log_returns = np.log(prices / prices.shift(1)).dropna()

    mu = log_returns.mean() * periods_per_year
    sigma = log_returns.std(ddof=1) * np.sqrt(periods_per_year)
    rho = log_returns.corr().iloc[0, 1]

    return {
        "mu1": float(mu.iloc[0]),
        "mu2": float(mu.iloc[1]),
        "sigma1": float(sigma.iloc[0]),
        "sigma2": float(sigma.iloc[1]),
        "rho": float(rho),
        "log_returns": log_returns
    }

In [ ]:
def build_three_states_from_moments(mu1, mu2, sigma1, sigma2, rho):
    """
    Construct a three-state return model matching mean, variance, and correlation.

    Assumes equal real-world probabilities (1/3 each).

    Parameters
    ----------
    mu1, mu2 : float
        Expected returns for each asset.
    sigma1, sigma2 : float
        Volatilities of each asset.
    rho : float
        Correlation between assets.

    Returns
    -------
    tuple
        (r1_states, r2_states, probabilities)
    """
    # Real-world probabilities are 1/3 under this construction of states
    p = np.array([1/3, 1/3, 1/3], dtype=float)

    # Stock 1 deviations
    d1 = sigma1 * np.sqrt(3 / 2)
    x = np.array([d1, 0.0, -d1])  # centered deviations for stock 1

    # Need y for stock 2 such that:
    # mean(y)=0
    # sum(y^2)/3 = sigma2^2
    # sum(x*y)/3 = rho*sigma1*sigma2
    #
    # Use y = a*x + z, where z is orthogonal to x and sums to zero.
    target_cov = rho * sigma1 * sigma2
    a = target_cov / sigma1**2

    # Basis vector orthogonal to x and summing to zero
    z_base = np.array([1.0, -2.0, 1.0])

    # divide by 3: from p
    var_x = np.sum(x**2) / 3.0
    var_z = np.sum(z_base**2) / 3.0

    residual_var = sigma2**2 - a**2 * var_x
    if residual_var < -1e-12:
        raise ValueError("Target correlation too extreme for this 3-state construction.")
    residual_var = max(residual_var, 0.0)

    b = np.sqrt(residual_var / var_z)
    y = a * x + b * z_base

    r1_states = mu1 + x
    r2_states = mu2 + y

    return r1_states, r2_states, p

In [ ]:
def solve_risk_neutral_probs_from_returns(r1_states, r2_states, risk_free_rate):
    """
    Solve for risk-neutral probabilities in a two-asset, three-state model.

    Parameters
    ----------
    r1_states, r2_states : array-like
        Returns of each asset in each state.
    risk_free_rate : float
        Risk-free rate.

    Returns
    -------
    np.ndarray
        Risk-neutral probabilities for the three states.
    """
    r1_states = np.asarray(r1_states, dtype=float)
    r2_states = np.asarray(r2_states, dtype=float)

    R1 = 1.0 + r1_states
    R2 = 1.0 + r2_states

    A = np.array([
        [1.0, 1.0, 1.0],
        R1,
        R2
    ], dtype=float)

    b = np.array([
        1.0,
        1.0 + risk_free_rate,
        1.0 + risk_free_rate
    ], dtype=float)

    q = np.linalg.solve(A, b)
    return q

In [ ]:
def two_stock_model(
    base_capital,
    w1,
    w2,
    r1_states,
    r2_states,
    S1_0,
    S2_0,
    X1,
    X2,
    risk_free_rate=0.05,
    p=np.array([1/3, 1/3, 1/3]),
):
    """
    Evaluate hedging strategies for a two-stock portfolio in a three-state model.

    This function compares three strategies:
    (1) Unhedged portfolio,
    (2) Hedging with individual put options on each stock,
    (3) Hedging with a basket put option on the combined portfolio.

    The model assumes a discrete three-state return structure for each stock and
    computes option prices using risk-neutral probabilities derived from the
    no-arbitrage condition.

    Parameters
    ----------
    base_capital : float
        Initial capital invested in the portfolio.
    w1 : float
        Portfolio weight allocated to stock 1 (must satisfy w1 + w2 = 1).
    w2 : float
        Portfolio weight allocated to stock 2.
    r1_states : array-like
        Returns of stock 1 in each of the three states.
    r2_states : array-like
        Returns of stock 2 in each of the three states.
    risk_free_rate : float
        Risk-free interest rate over the investment horizon.
    S1_0 : float
        Current price of stock 1.
    S2_0 : float
        Current price of stock 2.
    X1 : float
        Strike price of the put option on stock 1.
    X2 : float
        Strike price of the put option on stock 2.
    risk_free_rate : float
        Risk-free interest rate over the investment horizon (default is 5%).
    p : array-like, optional
        Real-world probabilities of each state (default is equal probabilities).

    Returns
    -------
    dict
        A dictionary containing:
        - "summary": pd.DataFrame
            Summary statistics for each strategy including financed cost,
            expected terminal value, worst-case value, and expected return.
        - "detail": pd.DataFrame
            State-by-state breakdown of returns, prices, and payoffs.
        - "spots": dict
            Initial stock prices.
        - "strikes": dict
            Individual and basket strike prices.
        - "option_prices": dict
            Prices of individual put options and total hedging costs.
        - "risk_neutral_probs": np.ndarray
            Risk-neutral probabilities for the three states.

    Notes
    -----
    - The portfolio is constructed using allocation based on the
      specified weights and initial capital.
    - Hedging costs are assumed to be financed at the risk-free rate and repaid
      at the end of the period.
    - Risk-neutral probabilities are computed by enforcing that discounted
      expected returns of both assets equal the risk-free rate.
    - The basket option strike is defined as the weighted sum of individual strikes.
    - Expected returns are computed using real-world probabilities (p), not
      risk-neutral probabilities.

    Raises
    ------
    ValueError
        If portfolio weights do not sum to 1.
    """
    r1_states = np.asarray(r1_states, dtype=float)
    r2_states = np.asarray(r2_states, dtype=float)
    p = np.asarray(p, dtype=float)

    if not np.isclose(w1 + w2, 1.0):
        raise ValueError("Portfolio weights w1 and w2 must sum to 1.")

    # Solve risk-neutral probabilities
    q = solve_risk_neutral_probs_from_returns(r1_states, r2_states, risk_free_rate)

    n1 = (w1 * base_capital) / S1_0
    n2 = (w2 * base_capital) / S2_0

    # Terminal stock prices in each state
    S1_T = S1_0 * (1.0 + r1_states)
    S2_T = S2_0 * (1.0 + r2_states)

    # Unhedged terminal portfolio values
    unhedged_terminal = n1 * S1_T + n2 * S2_T

    # Basket strike
    Xb = n1 * X1 + n2 * X2

    # Put payoffs
    put1_payoff = np.maximum(X1 - S1_T, 0.0)
    put2_payoff = np.maximum(X2 - S2_T, 0.0)

    two_put_payoff = n1 * put1_payoff + n2 * put2_payoff
    basket_put_payoff = np.maximum(Xb - unhedged_terminal, 0.0)

    # Option prices via risk-neutral probabilities
    disc = 1.0 / (1.0 + risk_free_rate)

    P1 = disc * np.dot(q, put1_payoff)
    P2 = disc * np.dot(q, put2_payoff)

    two_put_cost = n1 * P1 + n2 * P2
    basket_put_cost = disc * np.dot(q, basket_put_payoff)

    # Hedged terminal values
    two_put_terminal = unhedged_terminal + two_put_payoff
    basket_put_terminal = unhedged_terminal + basket_put_payoff

    # base stock investment = base_capital
    # hedge cost is financed to the end of the period
    financed_cost_unhedged = base_capital
    financed_cost_two_put = base_capital + (1.0 + risk_free_rate) * two_put_cost
    financed_cost_basket = base_capital + (1.0 + risk_free_rate) * basket_put_cost

    def make_row(name, terminal_values, financed_cost):
        expected_terminal = np.dot(p, terminal_values)
        worst_case = np.min(terminal_values)
        expected_return = (expected_terminal - financed_cost) / base_capital
        return {
            "Strategy": name,
            "Financed Cost": financed_cost,
            "Expected Terminal Value": expected_terminal,
            "Worst-Case Value": worst_case,
            "Expected Return": expected_return
    }

    summary = pd.DataFrame([
        make_row("Unhedged", unhedged_terminal, financed_cost_unhedged),
        make_row("Two puts", two_put_terminal, financed_cost_two_put),
        make_row("Basket puts", basket_put_terminal, financed_cost_basket),
    ])

    detail = pd.DataFrame({
        "State": ["U", "M", "D"],
        "Real Prob": p,
        "RN Prob": q,
        "r1_state": r1_states,
        "r2_state": r2_states,
        "S1_T": S1_T,
        "S2_T": S2_T,
        "Unhedged": unhedged_terminal,
        "Two-put payoff": two_put_payoff,
        "Basket-put payoff": basket_put_payoff,
        "Two-puts strategy": two_put_terminal,
        "Basket-put strategy": basket_put_terminal,
    })

    return {
        "summary": summary,
        "detail": detail,
        "spots": {"S1_0": S1_0, "S2_0": S2_0},
        "strikes": {"X1": X1, "X2": X2, "Xb": Xb},
        "option_prices": {
            "P1": P1,
            "P2": P2,
            "two_put_cost": two_put_cost,
            "basket_put_cost": basket_put_cost
        },
        "risk_neutral_probs": q
    }

In [ ]:
def run_two_stock_model(
    ticker1,
    ticker2,
    start="2020-01-01",
    end=None,
    risk_free_rate=0.05,
):
    """
    High-level wrapper that runs the two-stock hedging model using market data.

    This function handles data retrieval and parameter estimation before
    passing the resulting inputs to `two_stock_model`, which performs the
    portfolio and option pricing calculations.

    Specifically, it:
    - Downloads historical price data for two assets
    - Estimates return moments (mean, volatility, correlation)
    - Constructs a three-state return model
    - Retrieves option strike prices
    - Calls `two_stock_model` to evaluate hedging strategies

    Parameters
    ----------
    ticker1 : str
        Ticker symbol for the first stock.
    ticker2 : str
        Ticker symbol for the second stock.
    start : str, optional
        Start date for historical data (YYYY-MM-DD).
    end : str, optional
        End date for historical data. If None, uses latest available data.
    risk_free_rate : float, optional
        Risk-free interest rate over the investment horizon.

    Returns
    -------
    dict
        Output from `two_stock_model`, including summary statistics,
        state-level results, option prices, and risk-neutral probabilities.

    Notes
    -----
    This function separates data handling from model computation, allowing
    `two_stock_model` to remain a purely mathematical component.
    """
    prices = get_adjusted_prices([ticker1, ticker2], start=start, end=end)

    # $100 equal-dollar portfolio:
    # $50 in each stock
    w1 = 0.5
    w2 = 0.5
    base_capital = 100.0

    # Build 3 real-world states of equal probability from estimated moments
    moments = estimate_moments(prices)

    mu1 = moments["mu1"]
    mu2 = moments["mu2"]
    sigma1 = moments["sigma1"]
    sigma2 = moments["sigma2"]
    rho = moments["rho"]

    r1_states, r2_states, p = build_three_states_from_moments(mu1, mu2, sigma1, sigma2, rho)

    # Current spot prices
    S1_0 = float(prices.iloc[-1, 0])
    S2_0 = float(prices.iloc[-1, 1])

    # Strikes
    strike1_data = get_strike_price(ticker1)
    strike2_data = get_strike_price(ticker2)

    X1 = strike1_data["strike"]
    X2 = strike2_data["strike"]

    results  = two_stock_model(
        base_capital,
        w1,
        w2,
        r1_states,
        r2_states,
        S1_0,
        S2_0,
        X1,
        X2,
        risk_free_rate=risk_free_rate,
        p=p,
    )

    return results

# Results

In [ ]:
result = run_two_stock_model(
    ticker1="AAPL",
    ticker2="MSFT",
    start="2022-01-01",
    risk_free_rate=0.05
)

summary = result["summary"].copy()
summary["Financed Cost"] = summary["Financed Cost"].round(2)
summary["Expected Terminal Value"] = summary["Expected Terminal Value"].round(2)
summary["Worst-Case Value"] = summary["Worst-Case Value"].round(2)
summary["Expected Return"] = (100 * summary["Expected Return"]).round(2).astype(str) + "%"

print("\nRisk-neutral probabilities:")
print(result["risk_neutral_probs"])

print("\nSummary table:")
print(summary.to_string(index=False))

# Unit Tests

## two_stock_model

In [ ]:
def test_two_stock_model_expected_values():
    """
    Regression test for two_stock_model using a fixed three-state example.

    Verifies that:
    - Risk-neutral probabilities match expected values
    - Summary statistics (costs, returns, worst-case values) are correct
    - State-level payoffs and terminal values are computed correctly

    This test ensures that future changes do not alter the numerical
    outputs of the model.
    """
    mu1 = 0.09
    mu2 = 0.06

    base_capital = 100.0
    w1 = 0.5
    w2 = 0.5
    r1_states = [0.22, 0.10, -0.05]
    r2_states = [-0.05, 0.15, 0.08]
    S1_0 = 30
    S2_0 = 50
    X1 = S1_0 * (1 + (w1 * mu1 + w2 * mu2))
    X2 = S2_0 * (1 + (w1 * mu1 + w2 * mu2))
    risk_free_rate = 0.05
    p=np.array([1/3, 1/3, 1/3])

    result = two_stock_model(
        base_capital,
        w1,
        w2,
        r1_states,
        r2_states,
        S1_0,
        S2_0,
        X1,
        X2,
        risk_free_rate,
        p
    )

    expected_q = np.array([0.299479, 0.127604, 0.572917])

    assert np.allclose(
        result["risk_neutral_probs"],
        expected_q,
        atol=1e-6,
    )

    print("\nRisk-neutral probabilities:")
    print(result["risk_neutral_probs"])

    summary = result["summary"].copy()

    expected_summary = pd.DataFrame({
        "Strategy": ["Unhedged", "Two puts", "Basket puts"],
        "Financed Cost": [100.00, 105.45, 103.44],
        "Expected Terminal Value": [107.50, 111.67, 109.50],
        "Worst-Case Value": [101.50, 107.75, 107.50],
        "Expected Return": [0.0750, 0.0621, 0.0606],
    })

    np.testing.assert_allclose(
        summary["Financed Cost"],
        expected_summary["Financed Cost"],
        atol=1e-2,
    )

    np.testing.assert_allclose(
        summary["Expected Terminal Value"],
        expected_summary["Expected Terminal Value"],
        atol=1e-2,
    )

    np.testing.assert_allclose(
        summary["Worst-Case Value"],
        expected_summary["Worst-Case Value"],
        atol=1e-2,
    )

    np.testing.assert_allclose(
        summary["Expected Return"],
        expected_summary["Expected Return"],
        atol=1e-4,
    )

    summary["Financed Cost"] = summary["Financed Cost"].round(2)
    summary["Expected Terminal Value"] = summary["Expected Terminal Value"].round(2)
    summary["Worst-Case Value"] = summary["Worst-Case Value"].round(2)
    summary["Expected Return"] = (100 * summary["Expected Return"]).round(2).astype(str) + "%"

    print("\nSummary table:")
    print(summary.to_string(index=False))

    detail = result["detail"]

    np.testing.assert_allclose(
        detail["Unhedged"],
        [108.5, 112.5, 101.5],
        atol=1e-6,
    )

    np.testing.assert_allclose(
        detail["Two-put payoff"],
        [6.25, 0.0, 6.25],
        atol=1e-6,
    )

    np.testing.assert_allclose(
        detail["Basket-put payoff"],
        [0.0, 0.0, 6.0],
        atol=1e-6,
    )

    np.testing.assert_allclose(
        detail["Two-puts strategy"],
        [114.75, 112.50, 107.75],
        atol=1e-6,
    )

    np.testing.assert_allclose(
        detail["Basket-put strategy"],
        [108.5, 112.5, 107.5],
        atol=1e-6,
    )

    print("\nDetail:")
    print(result["detail"])

In [ ]:
test_two_stock_model_expected_values()